In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

c:\Users\mervy\Documents\NUS CS\Y3S2\web3 hackathon\quant_hackathon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 285,120


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,imbalance,imbalance_5,imbalance_15,trend_strength,vol_regime_ratio,is_trending,is_high_vol,mom_x_imb,mr_x_vol,trend_x_imb
0,2025-09-01 00:00:00+00:00,108246.36,108260.00,108210.66,108260.00,15.88924,2025-09-01 00:00:59.999999+00:00,1.719711e+06,2717,3.23174,...,-0.593217,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
1,2025-09-01 00:01:00+00:00,108260.00,108332.35,108259.99,108332.35,12.94030,2025-09-01 00:01:59.999999+00:00,1.401477e+06,1309,8.13811,...,0.257793,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
2,2025-09-01 00:02:00+00:00,108332.35,108332.35,108256.43,108256.44,25.92896,2025-09-01 00:02:59.999999+00:00,2.807727e+06,2136,0.53008,...,-0.959113,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
3,2025-09-01 00:03:00+00:00,108256.44,108282.43,108229.17,108229.18,18.99223,2025-09-01 00:03:59.999999+00:00,2.056101e+06,2344,8.31355,...,-0.124531,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
4,2025-09-01 00:04:00+00:00,108229.18,108229.18,108100.00,108100.00,12.05048,2025-09-01 00:04:59.999999+00:00,1.303485e+06,3790,2.20353,...,-0.634283,-0.41067,NaN,NaN,NaN,0,0,NaN,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 285,042
[info] optuna train rows: 182,426
[info] valid rows:        45,607
[info] test rows:         57,009


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-18 19:15:46,987] A new study created in memory with name: no-name-86e81f17-6fd4-4368-8651-bff89430b4eb


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [02:22<?, ?it/s]

  0%|          | 0/50 [02:22<?, ?it/s]

  0%|          | 0/50 [02:22<?, ?it/s]

[W 2026-03-18 19:18:09,477] Trial 0 failed with parameters: {'n_estimators': 300, 'max_depth': 19, 'min_samples_split': 10, 'min_samples_leaf': 30, 'max_features': 0.3, 'bootstrap': True} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\mervy\Documents\NUS CS\Y3S2\web3 hackathon\quant_hackathon\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\mervy\Documents\NUS CS\Y3S2\web3 hackathon\quant_hackathon\models.py", line 24, in _rf_objective
    model.fit(X_train, y_train)
  File "c:\Users\mervy\Documents\NUS CS\Y3S2\web3 hackathon\quant_hackathon\.venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\mervy\Documents\NUS CS\Y3S2\web3 hackathon\quant_hackathon\.venv\Lib\site-packages\sklearn\ensemble\_

In [ ]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

In [ ]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [ ]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

In [ ]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)

In [ ]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")

In [ ]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")